# Format LLM output

## Set up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH
import pandas as pd

raw_test_dir = BASE_PATH / "data/cleaned/question_answer/test"
raw_llm_output_dir = BASE_PATH / "data/test_data/raw_llm_outputs"
cleaned_test_export_dir = BASE_PATH / "data/test_data/cleaned"

In [6]:
base_cols = [
    "formatted_text",
    "S1_Desire_to_live",
    "S2_Suicidal_ideation",
    "S3_Self-harm_and_dangerous_behavior",
    "S4_Suicide_attempt",
    "S5_Mental_pain",
    "S6_Hopelessness",
    "S7_Helplessness",
    "S8_Worthlessness",
    "S9_Feeling_bad",
    "S10_Guilt",
    "S11_Sadness",
    "S12_Anxiety",
    "S13_Fear_of_failure",
    "S14_Restlessness",
    "S15_Irritability",
    "S16_Frustration",
    "S17_Emotion_regulation",
    "S18_Self-control",
    "S19_Mood_reactivity",
    "S20_Humour",
    "S21_Anhedonia",
    "S22_Emptiness_(feeling_empty)",
    "S23_Emotional_blunting_due_to_depression",
    "S24_Motivation",
    "S25_Interest",
    "S26_Incurability",
    "S27_Pessimism",
    "S28_Cognitive_distortion",
    "S29_Decision_making",
    "S30_Executive_dysfunctions",
    "S31_Cognitive_flexibility",
    "S32_Problem_solving",
    "S33_Memory_loss",
    "S34_Concentration",
    "S35_Time_management",
    "S36_Perception_of_time",
    "S37_Future_outlook",
    "S38_Rumination",
    "S39_Thoughts_clouding",
    "S40_Social_interest",
    "S41_Empathy",
    "S42_Fatigue",
    "S43_Energy",
    "S44_Weakness",
    "S45_Everything_is_an_effort",
    "S46_Psychomotor_retardation",
    "S47_Facial_expression",
    "S48_Mutism",
    "S49_Crying",
    "S50_Insomnia",
    "S51_Disturbed_sleep",
    "S52_Hypersomnia",
    "S53_Sexual_dysfunction",
    "S54_Loss_of_appetite_loss_of_weight",
    "S55_Pain_and_other_physical_symptoms",
    "S56_Self-recognition",
    "S57_Dissociative_symptoms",
    "S58_Self-esteem",
    "S59_Self-confidence",
    "S60_To_feel_misunderstood",
    "S61_Feeling_like_a_burden",
    "S62_Feeling_alone",
    "S63_Persecution",
    "S64_Psychotic_symptoms",
    "S65_To_get_out_of_bed",
    "S66_Daily_tasks",
    "S67_Self-care",
    "S68_Autonomy",
    "S69_Communicating_feelings",
    "S70_Social_isolation",
    "S71_Social_exclusion",
    "S72_Self-exclusion",
    "S73_Interpersonal_relationships",
    "S74_Family_life",
    "S75_Professional_responsibilities",
    "S76_Job_studies",
    "S77_Sick_leave",
    "S78_Administrative_financial_issues",
    "S79_Coping_with_life_events",
    "S80_Personal_development",
]

new_col_dict = {
    og_col: og_col.split("_")[0] for og_col in base_cols if og_col != "formatted_text"
}

In [ ]:
all_dfs = []
for file_path in raw_llm_output_dir.iterdir():
    # Extract meta-data to determine output path
    file_name_list = file_path.stem.split("_")
    shot_type = file_name_list[0]
    llm_model = file_name_list[1]
    date_created = file_name_list[2]
    impact_type = file_name_list[3]
    temp = file_name_list[4].split("temp")[1]
    output_type = file_name_list[5]
    if output_type != "SC":
        raise ValueError("Output type should be 'SC' (scale)")
    exp_path = (
        cleaned_test_export_dir
        / llm_model
        / f"temp_{temp}"
        / f"shot_{shot_type}"
        / "scale"
        / f"{impact_type}.parquet"
    )
    # Load and reformat raw file
    output_file = pd.read_csv(file_path)
    assert output_file.isna().sum().sum() == 0

    output_file_sub = output_file.drop("formatted_text", axis=1)
    text_cols = [col for col in output_file_sub.columns if col != "DAWBA_ID"]
    assert len(text_cols) == 80
    output_file_sub = output_file_sub.astype(float)
    output_file_sub = output_file_sub.rename(columns=new_col_dict)
    new_text_cols = list(new_col_dict.values())
    for col in new_text_cols:
        if not set(output_file_sub[col].unique()).issubset(set([0, 1, 2, 3, 4])):
            raise ValueError(
                f"ERROR: Values not in [0,1,2,3,4] {output_file_sub[col].unique()}"
            )
    output_file_sub[new_text_cols] = output_file_sub[new_text_cols] / 4
    for col in new_text_cols:
        if not set(output_file_sub[col].unique()).issubset(
            set([0, 0.25, 0.5, 0.75, 1])
        ):
            raise ValueError(
                f"ERROR: Values not in [0, 0.25, 0.5, 0.75, 1] {output_file_sub[col].unique()}"
            )
    base_df = pd.read_parquet(raw_test_dir / f"{impact_type}.parquet")
    assert set(output_file_sub["DAWBA_ID"]) == set(base_df["DAWBA_ID"])
    merged_df = pd.merge(base_df, output_file_sub, on="DAWBA_ID", how="inner")
    assert merged_df.isna().sum().sum() == 0
    # Export new df
    if exp_path.exists():
        exp_path.unlink()
        print(f"REMOVING file at path: {exp_path}")
    exp_path.parent.mkdir(exist_ok=True, parents=True)
    merged_df.to_parquet(exp_path)